# NetraGraph Intrusion Training
Upload or attach a labeled CSV/JSON/JSONL/TXT dataset. The target is detected from common names or can be set explicitly.

In [ ]:
import os, sys, shutil
from pathlib import Path
try:
    from google.colab import files, drive
    drive.mount('/content/drive')
except ImportError:
    files = None
DATA_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('/content/data')
OUT = Path('/kaggle/working/artifacts') if Path('/kaggle/input').exists() else Path('/content/artifacts')
PROJECT = Path('/content/NetraGraph') if Path('/content/NetraGraph').exists() else Path.cwd()
if not (PROJECT / 'backend/ml').exists() and files:
    uploaded = files.upload()
    source_zip = next((name for name in uploaded if name.endswith('.zip')), None)
    if source_zip:
        shutil.unpack_archive(source_zip, '/content/NetraGraph')
    PROJECT = Path('/content/NetraGraph')
if not (PROJECT / 'backend/ml').exists():
    raise FileNotFoundError('Upload a ZIP containing backend/ml for the portable trainer.')
sys.path.insert(0, str(PROJECT / 'backend'))
!pip install -q -r {PROJECT / 'requirements/requirements-colab.txt'}

In [ ]:
# Set DATASET to an attached Kaggle directory or uploaded/extracted Colab directory.
DATASET = DATA_ROOT
from ml.data.dataset_discovery import discover_files
files_found = discover_files(DATASET)
print('Discovered:', [str(p) for p in files_found])
if not files_found and files:
    uploaded = files.upload()
    for name, data in uploaded.items(): (DATA_ROOT / name).parent.mkdir(parents=True, exist_ok=True); (DATA_ROOT / name).write_bytes(data)
    files_found = discover_files(DATA_ROOT)

In [ ]:
from ml.training.train_all import train
bundle = train(DATASET, 'intrusion', output=OUT)
print('Bundle:', bundle)
import shutil
archive = shutil.make_archive(str(OUT / 'NetraGraph_intrusion_v1'), 'zip', root_dir=bundle.parent.parent, base_dir=bundle.parent.name)
print('Download:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Kaggle output is in /kaggle/working')